In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_TABLE = "supply_chain_dev.shipments.bronze_shipment_events"
SILVER_TABLE = "supply_chain_dev.shipments.silver_shipment_events"
QUARANTINE_TABLE = "supply_chain_dev.shipments.quarantine_shipment_events"

print("Config ready!")
print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")
print(f"Quarantine   : {QUARANTINE_TABLE}")

Config ready!
Reading from : supply_chain_dev.shipments.bronze_shipment_events
Writing to   : supply_chain_dev.shipments.silver_shipment_events
Quarantine   : supply_chain_dev.shipments.quarantine_shipment_events


In [0]:
# Read all records from Bronze
bronze_df = spark.table(BRONZE_TABLE)

# Identify corrupted records
corrupted_df = bronze_df.filter(
    col("booking_id").isNull() |
    col("status").isNull() |
    (col("weight_kg") < 0) |
    col("event_timestamp").isNull()
).withColumn("quarantine_reason", 
    when(col("booking_id").isNull(), "NULL_BOOKING_ID")
    .when(col("status").isNull(), "NULL_STATUS")
    .when(col("weight_kg") < 0, "INVALID_WEIGHT")
    .when(col("event_timestamp").isNull(), "MISSING_TIMESTAMP")
)

# Clean records = everything that is NOT corrupted
clean_df = bronze_df.filter(
    col("booking_id").isNotNull() &
    col("status").isNotNull() &
    (col("weight_kg") > 0) &
    col("event_timestamp").isNotNull()
).withColumn("event_timestamp", to_timestamp(col("event_timestamp"))) \
 .withColumn("estimated_arrival", to_timestamp(col("estimated_arrival")))

print(f"Total Bronze records : {bronze_df.count()}")
print(f"Clean records        : {clean_df.count()}")
print(f"Corrupted records    : {corrupted_df.count()}")

Total Bronze records : 466
Clean records        : 431
Corrupted records    : 35


In [0]:
# Write corrupted records to quarantine table
corrupted_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(QUARANTINE_TABLE)

print(f"Quarantine table created: {QUARANTINE_TABLE}")
print(f"Records quarantined: {corrupted_df.count()}")

# Preview what's in quarantine
display(spark.table(QUARANTINE_TABLE).select(
    "booking_id", "status", "weight_kg", 
    "event_timestamp", "quarantine_reason"
).limit(10))

Quarantine table created: supply_chain_dev.shipments.quarantine_shipment_events
Records quarantined: 35


booking_id,status,weight_kg,event_timestamp,quarantine_reason
CMA204078666,PORT_ARRIVED,-99.0,2024-01-21T12:00:00,INVALID_WEIGHT
MSC959629660,null,21059.82,2024-02-03T18:00:00,NULL_STATUS
HAP162795957,null,9401.12,2024-01-13T15:00:00,NULL_STATUS
HAP162795957,IN_TRANSIT,6281.89,null,MISSING_TIMESTAMP
COS186019028,BOOKING_CONFIRMED,-99.0,2024-01-31T00:00:00,INVALID_WEIGHT
COS207354053,BOOKING_CONFIRMED,2489.96,null,MISSING_TIMESTAMP
CMA434961542,VESSEL_DEPARTED,-99.0,2024-02-03T21:00:00,INVALID_WEIGHT
null,CARGO_RECEIVED,7725.26,2024-01-14T15:00:00,NULL_BOOKING_ID
MSC274648506,CUSTOMS_HOLD,-99.0,2024-01-25T02:00:00,INVALID_WEIGHT
EVG842800803,CUSTOMS_RELEASED,17763.53,null,MISSING_TIMESTAMP


In [0]:
# First create the Silver table from clean records
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_TABLE)

print(f"Silver table created!")
print(f"Records in Silver: {spark.table(SILVER_TABLE).count()}")

Silver table created!
Records in Silver: 431


In [0]:
from delta.tables import DeltaTable

# Simulate 5 incoming updates to existing records
updates_df = spark.table(SILVER_TABLE).limit(5) \
    .withColumn("status", lit("DELIVERED")) \
    .withColumn("weight_kg", lit(9999.99))

# MERGE — update if exists, insert if new
silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

silver_delta.alias("silver").merge(
    updates_df.alias("incoming"),
    "silver.event_id = incoming.event_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("MERGE complete!")

# Verify the updates landed
display(spark.table(SILVER_TABLE)
    .filter(col("status") == "DELIVERED")
    .filter(col("weight_kg") == 9999.99)
    .select("event_id", "booking_id", "status", "weight_kg")
    .limit(5))

MERGE complete!


event_id,booking_id,status,weight_kg
c5684ede-a617-4bbd-bcf8-f7ebb4d609f6,EVG330035022,DELIVERED,9999.99
1e98a0ee-a584-49a8-92ff-c0d5705de1a7,CMA204078666,DELIVERED,9999.99
9cbbf5f2-bc0b-40f7-991c-29d337de7765,CMA204078666,DELIVERED,9999.99
5927c26d-1721-457e-9b6d-6f910ba630d2,CMA204078666,DELIVERED,9999.99
8aaaa46a-20e7-4634-a749-e0e1439ccc76,CMA204078666,DELIVERED,9999.99


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-23T21:36:09.000Z,8479271986133632,ravigroverus@gmail.com,MERGE,"Map(predicate -> [""(event_id#14227 = event_id#14263)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1175587767927122),9df68ec7-9b4f-4342-a5ec-dd1fa0daa907,0623-204258-8vs7ytvj-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 6130, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 7369, materializeSourceTimeMs -> 1088, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3066, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3042)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-23T21:35:35.000Z,8479271986133632,ravigroverus@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1175587767927122),ca9bce7c-303b-4808-b6a1-00a54834e3dc,0623-204258-8vs7ytvj-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 431, numOutputBytes -> 24801)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


In [0]:
# Time travel — read Silver table BEFORE the MERGE happened
before_merge = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(SILVER_TABLE)

after_merge = spark.table(SILVER_TABLE)

print(f"Version 0 (before MERGE) DELIVERED count: {before_merge.filter(col('status') == 'DELIVERED').count()}")
print(f"Version 1 (after MERGE)  DELIVERED count: {after_merge.filter(col('status') == 'DELIVERED').count()}")


Version 0 (before MERGE) DELIVERED count: 0
Version 1 (after MERGE)  DELIVERED count: 5


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# CDC logic — get latest status per booking_id
window = Window.partitionBy("booking_id").orderBy(col("event_timestamp").desc())

current_state_df = spark.table("supply_chain_dev.shipments.silver_shipment_events") \
    .withColumn("row_num", row_number().over(window)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# Write as Gold table — current state of every shipment
current_state_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("supply_chain_dev.shipments.current_shipment_state")

print(f"Current state table created!")
print(f"Unique shipments tracked: {current_state_df.count()}")
display(current_state_df.select(
    "booking_id", "carrier", "status", "event_timestamp"
).limit(10))

Current state table created!
Unique shipments tracked: 80


booking_id,carrier,status,event_timestamp
CMA149203558,Evergreen,PORT_ARRIVED,2024-01-28T12:00:00.000Z
CMA204078666,CMA CGM,CUSTOMS_HOLD,2024-01-22T04:00:00.000Z
CMA220123666,COSCO,IN_TRANSIT,2024-02-03T13:00:00.000Z
CMA244193987,COSCO,OUT_FOR_DELIVERY,2024-01-23T23:00:00.000Z
CMA379340713,MSC,OUT_FOR_DELIVERY,2024-01-19T10:00:00.000Z
CMA434961542,CMA CGM,IN_TRANSIT,2024-02-05T14:00:00.000Z
CMA479759538,Hapag-Lloyd,OUT_FOR_DELIVERY,2024-01-21T15:00:00.000Z
CMA506448196,COSCO,IN_TRANSIT,2024-01-17T15:00:00.000Z
CMA510751046,MSC,VESSEL_DEPARTED,2024-01-26T21:00:00.000Z
CMA538642403,Evergreen,IN_TRANSIT,2024-01-10T22:00:00.000Z


In [0]:
display(spark.table("supply_chain_dev.shipments.current_shipment_state")
    .select("booking_id", "carrier", "origin_port", "destination_port", "status", "event_timestamp")
    .orderBy("carrier")
    .limit(15))

booking_id,carrier,origin_port,destination_port,status,event_timestamp
HAP107721109,CMA CGM,NLRTM,DEHAM,CUSTOMS_HOLD,2024-01-25T21:00:00.000Z
HAP384063364,CMA CGM,CNSHA,AEJEA,VESSEL_DEPARTED,2024-01-27T01:00:00.000Z
EVG739830322,CMA CGM,CNSHA,NLRTM,CUSTOMS_HOLD,2024-01-27T06:00:00.000Z
EVG776205431,CMA CGM,CNSHA,NLRTM,OUT_FOR_DELIVERY,2024-01-27T09:00:00.000Z
HAP155165334,CMA CGM,KRPUS,SGSIN,PORT_ARRIVED,2024-01-24T18:00:00.000Z
HAP570497096,CMA CGM,AEJEA,SGSIN,PORT_ARRIVED,2024-01-10T17:00:00.000Z
CMA204078666,CMA CGM,HKHKG,CNSHA,CUSTOMS_HOLD,2024-01-22T04:00:00.000Z
EVG334978881,CMA CGM,CNSHA,HKHKG,IN_TRANSIT,2024-01-16T09:00:00.000Z
COS914073098,CMA CGM,HKHKG,NLRTM,OUT_FOR_DELIVERY,2024-02-08T22:00:00.000Z
CMA604988121,CMA CGM,HKHKG,SGSIN,CUSTOMS_HOLD,2024-01-13T13:00:00.000Z
